<a href="https://colab.research.google.com/github/hastiabbasi/SLP-Data-Science-26/blob/An-exploration/nov_8_spaceship_titanic_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nov. 8 Spaceship Titanic Task

## Goal

This notebook is a continuation of the Nov. 1 feature engineering and model comparison work. This week, we are adding a **new machine learning algorithm** to see whether it can improve our validation accuracy and Kaggle submission.

The new model is **XGBoost**, a gradient-boosted tree model. It can run on a regular computer using the CPU, so no GPU is required. We will compare it with the Logistic Regression and Random Forest models from the Nov. 1 notebook.

You will also create charts that help us explain the model comparison and passenger patterns in our final presentation.

## What you will complete

1. Recreate the engineered dataset from Nov. 1.
2. Reuse the Logistic Regression and Random Forest baselines.
3. Build and train a new XGBoost model.
4. Compare all three models using the same validation split.
5. Select the strongest model and create a new Kaggle submission.
6. Create presentation-ready charts and write a short conclusion.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Required deliverables

Before the next meeting, complete the notebook and have:

- XGBoost trained successfully
- validation accuracy recorded for Logistic Regression, Random Forest, and XGBoost
- a comparison table and chart for all three models
- a selected final model based on validation accuracy
- a checked `nov_8_model_submission.csv` file
- the submission uploaded to Kaggle if possible
- a confusion matrix for the selected model
- an XGBoost feature-importance chart
- one passenger-pattern chart
- a short conclusion explaining whether the new model helped

The goal is not only to obtain a higher score. We also want to understand how a new algorithm compares with our earlier baselines.

## Before you begin

1. Make your own copy of this notebook.
2. Put `train.csv`, `test.csv`, and `sample_submission.csv` in one Google Drive folder.
3. Update `DATA_DIR` below.
4. Run the cells from top to bottom.

This notebook uses Google Drive so the files and chart images remain available when the Colab session is reopened. If you used the earlier `/content/train.csv` method, that method is still valid; you only need to adjust the loading section.

XGBoost is installed in the next section for this Colab session. It will run on the CPU and does not require a GPU.

## 1. Connect Google Drive and load the data

In [2]:
from google.colab import drive
drive.mount("/content/drive")

# Change this path to the folder containing your CSV files.
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/spaceship-titanic"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import pandas as pd

data_folder = Path(DATA_DIR)
train = pd.read_csv(data_folder / "train.csv")
test = pd.read_csv(data_folder / "test.csv")
sample_submission = pd.read_csv(data_folder / "sample_submission.csv")

print("Training data loaded:", train.shape)
print("Test data loaded:", test.shape)
print("Sample submission loaded:", sample_submission.shape)
display(train.head())

Training data loaded: (8693, 14)
Test data loaded: (4277, 13)
Sample submission loaded: (4277, 2)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 2. Install and import the machine learning and visualization tools

In [4]:
# XGBoost is a new model for this week's task.
!pip install -q xgboost

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix
from xgboost import XGBClassifier

## 3. Create a consistent validation split

All three models must use the same validation passengers. This makes the comparison fair. Do not change the `random_state` or validation size unless you have a specific reason and record it in your conclusion.

In [6]:
target = "Transported"
y = train[target].astype(int)

train_indices, valid_indices = train_test_split(
    train.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(train_indices))
print("Validation rows:", len(valid_indices))

Training rows: 6954
Validation rows: 1739


## 4. Recreate the engineered features

These are the features introduced in the Nov. 1 notebook:

- `TotalSpending`
- `CabinDeck`
- `CabinSide`
- `GroupSize`
- `IsAlone`

Run this section before training any model. XGBoost will use the same inputs as the earlier models so that the algorithm comparison is meaningful.

In [7]:
train_engineered = train.copy()
test_engineered = test.copy()

spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

for dataframe in [train_engineered, test_engineered]:
    dataframe["TotalSpending"] = dataframe[spending_columns].sum(
        axis=1,
        min_count=1
    )
    dataframe["CabinDeck"] = dataframe["Cabin"].str.split("/").str[0]
    dataframe["CabinSide"] = dataframe["Cabin"].str.split("/").str[2]
    dataframe["GroupId"] = dataframe["PassengerId"].str.split("_").str[0]

all_passenger_ids = pd.concat([
    train_engineered["PassengerId"],
    test_engineered["PassengerId"]
])
all_group_ids = all_passenger_ids.str.split("_").str[0]
group_id_counts = all_group_ids.value_counts()

for dataframe in [train_engineered, test_engineered]:
    dataframe["GroupSize"] = dataframe["GroupId"].map(group_id_counts)
    dataframe["IsAlone"] = (dataframe["GroupSize"] == 1).astype(int)

new_columns = ["TotalSpending", "CabinDeck", "CabinSide", "GroupSize", "IsAlone"]
display(train_engineered[new_columns].head())

,TotalSpending,CabinDeck,CabinSide,GroupSize,IsAlone
0,0.0,B,P,1,1
1,736.0,F,S,1,1
2,10383.0,A,S,2,0
3,5176.0,A,S,2,0
4,1091.0,F,S,1,1


## 5. Build the preprocessing and model function

Numerical missing values are filled with the median. Categorical missing values are filled with the most frequent value, then one-hot encoded. The same preprocessing is used for all three models.

In [8]:
def build_model(categorical_features, numerical_features, classifier):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

## 6. Prepare the model inputs

In [9]:
drop_columns = [target, "PassengerId", "Name", "Cabin", "GroupId"]

categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide"
]

numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "TotalSpending",
    "GroupSize",
    "IsAlone"
]

X_engineered = train_engineered.drop(columns=drop_columns)
print("Input columns:", list(X_engineered.columns))

Input columns: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpending', 'CabinDeck', 'CabinSide', 'GroupSize', 'IsAlone']


## 7. Recreate the Nov. 1 baseline models

The following models are our comparison baselines. Run this section without changing it so we can tell whether the new XGBoost model performs better than the earlier approaches.

In [10]:
logistic_model = build_model(
    categorical_features,
    numerical_features,
    LogisticRegression(max_iter=1000)
)

random_forest_model = build_model(
    categorical_features,
    numerical_features,
    RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
)

logistic_model.fit(X_engineered.loc[train_indices], y.loc[train_indices])
logistic_validation_predictions = logistic_model.predict(X_engineered.loc[valid_indices])
logistic_accuracy = accuracy_score(y.loc[valid_indices], logistic_validation_predictions)

random_forest_model.fit(X_engineered.loc[train_indices], y.loc[train_indices])
random_forest_validation_predictions = random_forest_model.predict(X_engineered.loc[valid_indices])
random_forest_accuracy = accuracy_score(y.loc[valid_indices], random_forest_validation_predictions)

print("Logistic Regression validation accuracy:", logistic_accuracy)
print("Random Forest validation accuracy:", random_forest_accuracy)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression validation accuracy: 0.7901092581943646
Random Forest validation accuracy: 0.8050603795284647


## 8. Build and train the new XGBoost model

XGBoost builds many small decision trees sequentially. Each new tree focuses on mistakes made by the earlier trees. The settings below are intentionally modest so the model runs on a normal computer.

### Your task

Complete the TODOs below:

1. Create an `XGBClassifier`.
2. Put it inside the existing `build_model` preprocessing pipeline.
3. Fit it only on `train_indices`.
4. Predict only on `valid_indices`.
5. Calculate and record its validation accuracy.

Suggested starting settings:

```python
XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
```

Do not use `use_label_encoder`; it is no longer needed in current XGBoost versions.

In [11]:
# TODO 1: create the XGBClassifier using the suggested settings above.
xgb_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# TODO 2: build a pipeline using the same preprocessing function.
xgboost_model = build_model(
    categorical_features,
    numerical_features,
    xgb_classifier
)

# TODO 3: fit on the training split and predict on the validation split.
xgboost_model.fit(X_engineered.loc[train_indices],y.loc[train_indices])
xgboost_validation_predictions = xgboost_model.predict(X_engineered.loc[valid_indices])

# TODO 4: calculate the validation accuracy.
xgboost_accuracy = accuracy_score(y.loc[valid_indices],xgboost_validation_predictions)

# Print your result after completing the TODOs.
print("XGBoost validation accuracy:", xgboost_accuracy)

XGBoost validation accuracy: 0.8102357676825762


## 9. Compare all three models

Add the new XGBoost results to the existing comparison table. The table and chart should contain exactly three rows: Logistic Regression, Random Forest, and XGBoost.

### Your task

- Replace the TODO section with a `results` DataFrame.
- Sort it from highest to lowest validation accuracy.
- Display it.
- Save it as `nov_8_model_comparison.csv`.
- Write one sentence below the output explaining which model performed best on this validation split.

In [13]:
# TODO: create a results DataFrame containing all three models and their accuracies.
# It should have two columns: "Model" and "Validation Accuracy".
# Include Logistic Regression, Random Forest, and XGBoost.
# Sort from highest to lowest validation accuracy and display the result.
# Save the completed table as nov_8_model_comparison.csv.

model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Validation Accuracy": [logistic_accuracy, random_forest_accuracy, xgboost_accuracy]
}).sort_values("Validation Accuracy", ascending=False).reset_index(drop=True)

comparision_path = data_folder/"nov_8_model_comparison.csv"
model_results.to_csv(comparision_path, index=False )

#The XGBoost have the highest accuracy at 81% with the engineered features

## 10. Select the best model and create the new submission

Use the validation results to select the strongest model. Then retrain that model on **all** labeled training data before predicting the test passengers.

### Your task

Complete the TODOs so that:

- XGBoost can be selected if it has the highest accuracy.
- `best_model_name` stores the selected model's name.
- `best_validation_predictions` stores the selected model's validation predictions.
- the selected model is fit on all of `X_engineered` and `y`.
- the file is saved as `nov_8_model_submission.csv`.

In [ ]:
# TODO: select the model with the highest validation accuracy.
# Hint: create dictionaries that connect each model name to its model,
# validation predictions, and validation accuracy. Then use max(..., key=...).

# prediction_dictionary = {...}
# accuracy_dictionary = {...}
# best_model_name = ...
# best_model = ...
# best_validation_predictions = ...

# TODO: fit the selected model on all of X_engineered and y.
# TODO: predict the test data and create a DataFrame with PassengerId and Transported.
# TODO: save the submission as nov_8_model_submission.csv.

## 11. Check the new submission

Run these checks before uploading the file to Kaggle.

In [ ]:
print("Submission shape:", submission.shape)
print("Expected test rows:", len(test))
print("Submission columns:", list(submission.columns))
print("Missing predictions:", submission["Transported"].isna().sum())

assert len(submission) == len(test)
assert list(submission.columns) == ["PassengerId", "Transported"]
assert submission["Transported"].isna().sum() == 0

print("All submission checks passed.")

## 12. Create a model accuracy chart

Use the `results` DataFrame to create a bar chart comparing all three models. This is one of the figures we can use in the final presentation.

### Your task

- Create a bar chart from `results`.
- Add a clear title and axis labels.
- Make sure all three model names are visible.
- Save it as `nov_8_model_accuracy_comparison.png`.

In [ ]:
# TODO: create a bar chart from the results DataFrame.
# The chart should include all three models, have a title and axis labels,
# and be saved as nov_8_model_accuracy_comparison.png.

# Hint:
# accuracy_plot = results.plot(
#     kind="bar", x="Model", y="Validation Accuracy",
#     legend=False, figsize=(8, 4)
# )
# Add labels, use plt.tight_layout(), save the figure, and display it.

## 13. Create a confusion matrix for the selected model

A confusion matrix shows which validation passengers were classified correctly and incorrectly. The rows represent actual values, and the columns represent predicted values.

### Your task

Use `best_validation_predictions` and save the figure as `nov_8_confusion_matrix.png`.

In [ ]:
# TODO: create a confusion matrix using the selected model's validation predictions.
# Use y.loc[valid_indices] as the true labels and
# best_validation_predictions as the predicted labels.
# Save the figure as nov_8_confusion_matrix.png.

# Hint: use confusion_matrix(...) and ConfusionMatrixDisplay(...).

## 14. Create an XGBoost feature-importance chart

This chart focuses on the new model. Feature importance measures which processed input columns contributed most to XGBoost's decisions. Because categorical variables were one-hot encoded, some feature names will include category values.

### Your task

Use the fitted `xgboost_model` pipeline to:

1. get the processed feature names from the preprocessor
2. get the classifier's `feature_importances_`
3. combine them into a DataFrame
4. display the top 12 features in a horizontal bar chart
5. save the chart as `nov_8_xgboost_feature_importance.png`

Hint: the relevant objects are `xgboost_model.named_steps["preprocessor"]`, `xgboost_model.named_steps["classifier"]`, and `get_feature_names_out()`.

In [ ]:
# TODO: create the XGBoost feature-importance table and chart.
# Retrieve the preprocessor and classifier from xgboost_model.
# Then use get_feature_names_out() and feature_importances_.
# Display the top 12 features in a horizontal bar chart.
# Save the figure as nov_8_xgboost_feature_importance.png.

# Hint:
# xgb_preprocessor = xgboost_model.named_steps["preprocessor"]
# xgb_classifier_fitted = xgboost_model.named_steps["classifier"]
# feature_names = xgb_preprocessor.get_feature_names_out()
# feature_importances = xgb_classifier_fitted.feature_importances_

## 15. Create one passenger-pattern chart

This chart helps explain a pattern in the dataset independently of the model. Keep it in the notebook because it may be useful for the presentation.

### Your task

Run the example below and save the chart. In your conclusion, describe what the chart shows without claiming that it proves cause and effect.

In [ ]:
# TODO: create one passenger-pattern chart.
# Compare transportation rates by CryoSleep status, or choose another
# meaningful categorical variable. Add a clear title and axis labels.
# Save the chart as nov_8_transportation_by_cryosleep.png.

# Hint for a CryoSleep chart:
# rates = pd.crosstab(train["CryoSleep"], train["Transported"], normalize="index")
# rates.plot(kind="bar", figsize=(7, 4))
# Use plt.tight_layout(), save the figure, and display it.

## 16. Write the final project conclusion

Complete this section in your own words after running the notebook. Include:

- the validation accuracy of all three models
- whether XGBoost improved on the Nov. 1 baselines
- the model selected for the final submission and why
- one important feature-importance observation
- one pattern shown by the passenger chart
- one limitation or future improvement

Suggested structure:

> We continued our Nov. 1 work by comparing Logistic Regression and Random Forest with the new XGBoost algorithm. The best validation accuracy was ___, achieved by ___. XGBoost did/did not improve on our earlier models because ___. The feature-importance chart suggests ___. The passenger-pattern chart shows ___. One limitation is ___, so a future improvement could be ___ .

## Final checklist

- [ ] XGBoost was installed and trained successfully.
- [ ] Logistic Regression, Random Forest, and XGBoost validation accuracies were recorded.
- [ ] The three-model comparison table was created and saved.
- [ ] The final model was selected using validation accuracy.
- [ ] `nov_8_model_submission.csv` was created and checked.
- [ ] The new submission was uploaded to Kaggle if possible.
- [ ] The three-model accuracy chart was saved.
- [ ] The selected-model confusion matrix was saved.
- [ ] The XGBoost feature-importance chart was saved.
- [ ] The passenger-pattern chart was saved.
- [ ] The conclusion explains whether the new algorithm helped.
- [ ] The charts and results are ready to discuss in the final presentation.